#### Building basic Jarvis 01 from scratch, then later on moving onto papers and research grade shit

In [1]:
import json, math, platform, re, subprocess
from datetime import datetime
from pathlib import Path

##### Basic output function for JARVIS responses

In [2]:
def jarvis_speak(message: str) -> None:
    print(f"J.A.R.V.I.S says: {message}")

jarvis_speak("Hello! I am J.A.R.V.I.S, your personal assistant.")

J.A.R.V.I.S says: Hello! I am J.A.R.V.I.S, your personal assistant.


##### Time and date utility functions

In [3]:
def get_current_time():
    now = datetime.now()
    return now.strftime("%H:%M:%S")

def get_current_date():
    today = datetime.now()
    return today.strftime("%d-%m-%Y")

get_current_time()
get_current_date()

'22-08-2026'

##### Rule-based intent detection from user text

In [4]:
def detect_intent(text: str) -> str:

    text = text.lower().strip()

    if any(x in text for x in ["hello", "hi", "hey"]):
        return "greeting"
    if "who am i" in text or "what is my name" in text:
        return "memory_read"
    if "time" in text:
        return "get_current_time"
        return get_current_time()
    if "date" in text or "today" in text:
        return "date"
        return datetime.now().strftime("%d-%m-%Y")
    if any(x in text for x in ["calculate", "compute", "what is"]):
        return "calculator"
    if "remember" in text or "my name is" in text:
        return "memory_write"
    if text.startswith(("open ", "launch ")):
        return "open_app"
    if text in {"exit", "quit", "bye", "shutdown"}:
        return "exit"
    return "unknown"

for x in ["Hello", "What time is it?", "Calculate 5*8", "My name is Sparsh",
          "Who am I?", "Open VS Code", "quit"]:
    print(f"{x:22} -> {detect_intent(x)}")


Hello                  -> greeting
What time is it?       -> get_current_time
Calculate 5*8          -> calculator
My name is Sparsh      -> memory_write
Who am I?              -> memory_read
Open VS Code           -> open_app
quit                   -> exit


##### Safe arithmetic calculator tool

In [5]:
_ALLOWED = re.compile(r"^[0-9+\-*/().%\s]+$")

def calculate(expression: str) -> str:  # calculator function
    expression = re.sub(
        r"^(calculate|compute|what is)\s+",
        "",
        expression.strip(),
        flags=re.IGNORECASE
    ).strip()

    if not _ALLOWED.fullmatch(expression):
        return "I can only calculate basic arithmetic."

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        if isinstance(result, (int, float)) and math.isfinite(float(result)):
            return str(result)
        return "The result is not finite."
    except Exception:
        return "I could not calculate that."

##### Cross-platform application launcher

In [6]:
# Application Commands

APP_COMMANDS = {
    "linux": {
        "terminal": ["x-terminal-emulator"],
        "cmd": ["x-terminal-emulator"],
        "calculator": ["gnome-calculator"],
        "vscode": ["code"],
        "vs code": ["code"],
        "chrome": ["google-chrome"],
        "firefox": ["firefox"],
        "sublime": ["subl"],
        "sublime text": ["subl"],
        "spotify": ["spotify"],
        "vlc": ["vlc"],
        "notepad": ["gedit"],
        "paint": ["gimp"],
        "word": ["libreoffice", "--writer"],
        "excel": ["libreoffice", "--calc"],
        "powerpoint": ["libreoffice", "--impress"],
    }
}

def open_application(app_name: str) -> str:
    system = platform.system().lower()
    system_key = {"linux":"linux"}.get(system)

    if not system_key:
        return f"Unsupported OS: {platform.system()}"

    name = re.sub(r"^(open|launch)\s+", "", app_name.strip(), flags=re.I).lower()
    command = APP_COMMANDS[system_key].get(name)

    if not command:
        return f"I don't have a configured command for '{app_name}'."

    try:
        subprocess.Popen(command)
        return f"Opening {app_name}."
    except FileNotFoundError:
        return f"{app_name} is not installed or not on PATH."
    except Exception as e:
        return f"Could not open {app_name}: {e}"

##### JSON-based persistent memory load/save

In [7]:
MEMORY_FILE = Path("jarvis_memory.json")

def load_memory():  ## Load memory from a JSON file
    if not MEMORY_FILE.exists():
        return {}
    try:
        return json.loads(MEMORY_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return {}

def save_memory(memory):  ## Save memory to a JSON file
    MEMORY_FILE.write_text(
        json.dumps(memory, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

memory = load_memory()

def remember(key, value):  ## Remember a key-value pair in memory
    memory[key] = value
    save_memory(memory)
    return f"I'll remember that {key} is {value}."

def recall(key):  ## Recall a value from memory by key
    return memory.get(key)

##### Handlers for writing and reading memory facts

In [8]:
def handle_memory_write(text):   ## Handle memory write commands
    text = text.strip()

    match = re.fullmatch(r"my name is (.+)", text, flags=re.I)
    if match:
        return remember("name", match.group(1).strip().title())

    match = re.fullmatch(r"remember that my (.+?) is (.+)", text, flags=re.I)
    if match:
        return remember(match.group(1).strip(), match.group(2).strip())

    return "Tell me what you want me to remember."

def handle_memory_read(text):  ## Handle memory read commands
    if text.lower().strip() in {"who am i", "what is my name"}:
        name = recall("name")
        return f"Your name is {name}." if name else "I don't know your name yet."
    return "I can recall basic personal facts."

##### Central command router dispatching intents to handlers

In [9]:
def handle_command(text):
    intent = detect_intent(text)

    if intent == "greeting":
        jarvis_speak("Hello. How can I help?")

    elif intent == "time":
        jarvis_speak(f"The current time is {get_time()}.")

    elif intent == "date":
        jarvis_speak(f"Today is {get_date()}.")

    elif intent == "calculator":
        jarvis_speak(calculate(text))

    elif intent == "memory_write":
        jarvis_speak(handle_memory_write(text))

    elif intent == "memory_read":
        jarvis_speak(handle_memory_read(text))

    elif intent == "open_app":
        app = re.sub(r"^(open|launch)\s+", "", text.strip(), flags=re.I)
        jarvis_speak(open_application(app))

    elif intent == "exit":
        jarvis_speak("Shutting down. Goodbye.")
        return False

    else:
        jarvis_speak("I don't understand that yet.")

    return True

##### Main loop running JARVIS interactively

In [10]:
def run_jarvis():
    jarvis_speak("Systems online. JARVIS is ready.")

    while True:
        try:
            command = input("YOU: ").strip()
            if not command:
                continue
            if not handle_command(command):
                break
        except KeyboardInterrupt:
            print()
            jarvis_speak("Keyboard interrupt detected. Shutting down.")
            break
        except EOFError:
            print()
            jarvis_speak("Input stream closed. Shutting down.")
            break

run_jarvis()

J.A.R.V.I.S says: Systems online. JARVIS is ready.
J.A.R.V.I.S says: Opening calculator.



(gnome-calculator:95335): Adwaita-WARNING **: 03:27:16.030: The resource style-dark.css is deprecated and shouldn't be used anymore. Use style.css with media queries instead.

(gnome-calculator:95335): Adwaita-WARNING **: 03:27:16.030: The resource style-hc.css is deprecated and shouldn't be used anymore. Use style.css with media queries instead.



J.A.R.V.I.S says: Keyboard interrupt detected. Shutting down.
